# TUGAS MANDIRI

## Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7 & 9) ingin membangun dashboard pemantauan pesanan real-time untuk mengetahui performa tiap cabang setiap saat, bukan hanya laporan bulanan seperti sebelumnya. Anda ditugaskan membangun purwarupa (prototype) pipeline streaming yang mensimulasikan kebutuhan ini.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
import os
import shutil
import time
import pandas as pd
from datetime import datetime

# Buat SparkSession dengan shuffle partitions 4
spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession Tugas 10 Siap. Versi Spark:", spark.version)

26/11/11 18:34:25 WARN Utils: Your hostname, rindani-Latitude-3410 resolves to a loopback address: 127.0.1.1; using 192.168.1.9 instead (on interface wlp0s20f3)
26/11/11 18:34:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 18:34:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession Tugas 10 Siap. Versi Spark: 3.5.9


### A. Menyiapkan Streaming Source 

Definisikan skema eksplisit (StructType) untuk data pesanan dengan kolom: pesanan_id (string), nama_cabang (string), nama_menu (string), qty (integer), harga_satuan (integer), waktu_pesan (timestamp). Buat folder simulasi streaming baru bernama tugas10_stream, lalu definisikan readStream dari folder tersebut.

In [2]:
# Menyiapkan folder simulasi dan checkpoint
folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

# Skema eksplisit pesanan Kopi Nusantara
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

# Menyiapkan readStream
df_stream_tugas = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

print("Streaming source Bagian A berhasil disiapkan.")

Streaming source Bagian A berhasil disiapkan.


### B. Agregasi per Cabang 

Tambahkan kolom turunan total_harga (qty x harga_satuan) pada df_stream, lalu buat agregasi groupBy("nama_cabang") yang menghitung total total_harga dan jumlah pesanan. Jalankan sebagai streaming query dengan sink memory, queryName="agregasi_cabang".

In [3]:
# Tambahkan kolom turunan total_harga (qty x harga_satuan)
df_dengan_total = df_stream_tugas.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi per nama_cabang
agregasi_cabang = df_dengan_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Jalankan streaming query dengan memory sink
query_cabang = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Streaming query Bagian B (agregasi_cabang) dimulai.")

Streaming query Bagian B (agregasi_cabang) dimulai.


### C. Agregasi Berbasis Window Waktu 

Buat streaming query kedua (definisikan ulang readStream dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total total_harga berdasarkan window waktu 2 menit dari kolom waktu_pesan. Gunakan sink memory, queryName="agregasi_waktu".

In [4]:
# Membaca ulang streaming source yang sama untuk query kedua
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

df_waktu_dengan_total = df_stream_waktu.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi berdasarkan window waktu 2 menit dari kolom waktu_pesan
agregasi_waktu = df_waktu_dengan_total.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Jalankan streaming query dengan memory sink
query_waktu = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Streaming query Bagian C (agregasi_waktu) dimulai.")

Streaming query Bagian C (agregasi_waktu) dimulai.


### D. Menguji dengan Data Simulasi

Kirim minimal 3 batch data (masing-masing batch berupa satu berkas CSV baru dengan jeda time.sleep() di antaranya) ke folder tugas10_stream, dengan kombinasi nama_cabang yang bervariasi. Setelah setiap batch, tampilkan hasil agregasi_cabang untuk membuktikan angka totalnya terus terakumulasi dengan benar.

### E. Sink Berbasis Berkas (Tantangan) 

Selain memory sink, Structured Streaming juga mendukung menulis hasil DataFrame mentah (bukan hasil agregasi) ke berkas secara berkelanjutan. Buat streaming query ketiga yang membaca dari df_stream, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal tugas10_output dalam format Parquet, menggunakan:

query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (!ls tugas10_output) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan spark.read.parquet().

